# Synthetic Enhanced Dataset Pipeline

This notebook runs the standalone dataset generator pipeline. 
It downloads pre-processed CSV datasets, replaces malicious IPs with realistic IPs from Threat Intelligence feeds, and standardizes public benign IPs.

1. **Setup** — ML stack.
2. **Configure** target days and run parameters
3. **Run** pipeline


## Configuration


In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Override settings parameters dynamically if needed
import configs.settings as _s

DAYS = [
    'Thursday-01-03-2018',
    'Friday-02-03-2018',
]
FORCE_RERUN = False
SAMPLE_SIZE = 500000

_s.DAYS = DAYS
_s.SAMPLE_SIZE = SAMPLE_SIZE

print('Running for days:', DAYS)


## Run Pipeline


In [ ]:
from core.ingestion import Ingestion
from core.preprocessing import preprocess, clean_temp

PIPELINE_ROOT = Path('.').resolve()

ing = Ingestion(base_dir=PIPELINE_ROOT)
raw = ing.run(days=DAYS, force_rerun=FORCE_RERUN)
raw_df = raw['dataframe']

preproc_df = preprocess(
    raw_df,
    sample_size=SAMPLE_SIZE,
    cache=_s.CACHE_ENABLED,
    cache_dir=_s.CACHE_DIR,
)

clean_temp(PIPELINE_ROOT)

print('\nPipeline Terminated!')
print('Total records:', len(raw_df))


## Results Verification


In [ ]:
if not raw_df.empty:
    print('=== Label distribution ===')
    print(raw_df['Label'].value_counts().to_string())
    
    if 'Src IP' in raw_df.columns:
        print('\n=== Sample from IPs (Modified) ===')
        display(raw_df[['Src IP', 'Dst IP', 'Label']].sample(min(10, len(raw_df))))
else:
    print('No results fetched.')
